In [1]:
import torch
import torch.nn as nn


from torchinfo import summary

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))
!nvidia-smi

2.10.0+cu128
12.8
Tesla T4
Mon Jun 29 10:02:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------

In [2]:
try:
    import timm
except ImportError:
    !pip install -q timm --no-deps

In [3]:
MODEL_NAME = "efficientnet_b0"

HIDDEN_COLOR = 256

HIDDEN_TEXTURE = 128

HIDDEN_BINARY = 512

NUM_HAIR_COLOR = 5

NUM_HAIR_TEXTURE = 2

NUM_BINARY = 15

DROPOUT = 0.3

In [8]:


class AvatarAttributeModel(nn.Module):

    def __init__(self):

        super().__init__()

        # ----------------------------
        # Backbone
        # ----------------------------

        self.backbone = timm.create_model(

            MODEL_NAME,

            pretrained=True,

            num_classes=0,

            global_pool="avg"

        )

        feature_dim = self.backbone.num_features

        # ----------------------------
        # Shared Dropout
        # ----------------------------

        self.dropout = nn.Dropout(DROPOUT)

        # ----------------------------
        # Heads
        # ----------------------------

        self.hair_color = nn.Sequential(

            nn.Linear(feature_dim, HIDDEN_COLOR),

            nn.BatchNorm1d(HIDDEN_COLOR),

            nn.ReLU(inplace=True),

            nn.Dropout(0.3),

            nn.Linear(HIDDEN_COLOR, NUM_HAIR_COLOR)

        )

        self.hair_texture = nn.Sequential(

            nn.Linear(feature_dim,HIDDEN_TEXTURE),

            nn.BatchNorm1d(HIDDEN_TEXTURE),

            nn.ReLU(inplace=True),

            nn.Dropout(0.3),

            nn.Linear(HIDDEN_TEXTURE,NUM_HAIR_TEXTURE)

        )

        self.binary = nn.Sequential(

            nn.Linear(feature_dim,HIDDEN_BINARY),

            nn.BatchNorm1d(HIDDEN_BINARY),

            nn.ReLU(inplace=True),

            nn.Dropout(0.3),
            
            nn.Linear(HIDDEN_BINARY,NUM_BINARY)

        )

    def forward(self, x):

        features = self.backbone(x)

        features = self.dropout(features)

        return {

            "hair_color":
                self.hair_color(features),

            "hair_texture":
                self.hair_texture(features),

            "binary":
                self.binary(features)

        }

In [9]:
model = AvatarAttributeModel()

model

AvatarAttributeModel(
  (backbone): EfficientNet(
    (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (act1): SiLU(inplace=True)
            (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (gate): Sigmoid()
          )
          (co

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AvatarAttributeModel().to(device)
model.eval()

dummy = torch.randn(2, 3, 224, 224).to(device)

with torch.no_grad():
    outputs = model(dummy)

for name, tensor in outputs.items():
    print(name, tensor.shape)

hair_color torch.Size([2, 5])
hair_texture torch.Size([2, 2])
binary torch.Size([2, 15])


In [11]:
torch.save(
    model.state_dict(),
    "avatar_model_initial.pth"
)